In [0]:
%pip install Faker

dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

current = Path.cwd()
repo_root = None

for path in [current, *current.parents]:
    if (
        (path / "src" / "data_generator").exists()
        and
        (path / "databricks" / "ingestion" / "bronze_writer.py").exists()
    ):
        repo_root = path
        break

if repo_root is None:
    raise RuntimeError("Could not find AI-Analytics-Data-Platform repo root")

for import_path in [
    repo_root,
    repo_root / "databricks",
]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

print("Repo root:", repo_root)

In [0]:
import importlib
from datetime import date

import ingestion.delta_state as delta_state
import ingestion.bronze_writer as bronze_writer
import src.data_generator.daily_generation as daily_generation

importlib.reload(delta_state)
importlib.reload(bronze_writer)
importlib.reload(daily_generation)

SIMULATION_DATE = date(2026, 8, 23)
SEED = 42

print("Modules loaded")

In [0]:
state = delta_state.load_generator_state(spark)

clean_records = daily_generation.generate_daily_records(
    state,
    SIMULATION_DATE,
    SEED,
)

validation = daily_generation.validate_daily_records(
    clean_records,
    state,
    SIMULATION_DATE,
)

print(validation)

In [0]:
bronze_records, manifest = daily_generation._bronze_records(
    clean_records,
    SEED,
    SIMULATION_DATE,
    state,
)

for entity, rows in bronze_records.items():
    print(f"{entity}: {len(rows):,}")

print("\nInjected quality issues:", len(manifest))

In [0]:
summary = bronze_writer.validate_bronze_day(
    spark,
    bronze_records,
    SIMULATION_DATE,
)

for row in summary:
    print(
        f"{row['entity']}: "
        f"input={row['input_physical_row_count']:,}, "
        f"validated={row['validated_physical_row_count']:,}, "
        f"status={row['status']}, "
        f"write_attempted={row['write_attempted']}"
    )

In [0]:
for row in summary:
    assert row["status"] == "VALIDATED"
    assert row["write_attempted"] is False
    assert (
        row["input_physical_row_count"]
        == row["validated_physical_row_count"]
    )

print("PASS: all Aug 23 Bronze batches match live Delta schemas")
print("PASS: no writes were attempted")

In [0]:
TEST_SCHEMA = "ai_analytics.bronze_writer_test"
TEST_TABLE = f"{TEST_SCHEMA}.messages"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TEST_SCHEMA}")

spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")

spark.sql(f"""
    CREATE TABLE {TEST_TABLE}
    LIKE ai_analytics.bronze.messages
""")

print("Created:", TEST_TABLE)
print("Initial rows:", spark.table(TEST_TABLE).count())

In [0]:
production_schema = spark.table(
    "ai_analytics.bronze.messages"
).schema

test_schema = spark.table(
    TEST_TABLE
).schema

assert production_schema == test_schema

print("PASS: test table schema matches production Bronze messages")

In [0]:
messages_count = len(bronze_records["messages"])

first_write = bronze_writer.append_bronze_entity(
    spark,
    "messages",
    bronze_records["messages"],
    SIMULATION_DATE,
    target_schema=TEST_SCHEMA,
)

print(first_write)

count_after_first = spark.table(TEST_TABLE).count()

print("Expected rows:", messages_count)
print("Rows after first write:", count_after_first)

assert count_after_first == messages_count

print("PASS: first append wrote the complete physical Bronze batch")

In [0]:
from pyspark.sql import functions as F

duplicate_ids = (
    spark.table(TEST_TABLE)
    .groupBy("message_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_id_count = duplicate_ids.count()

print("Repeated message IDs:", duplicate_id_count)

assert duplicate_id_count > 0

display(duplicate_ids.limit(10))

print("PASS: intentional physical duplicates were preserved")

In [0]:
second_write = bronze_writer.append_bronze_entity(
    spark,
    "messages",
    bronze_records["messages"],
    SIMULATION_DATE,
    target_schema=TEST_SCHEMA,
)

print(second_write)

count_after_retry = spark.table(TEST_TABLE).count()

print("Rows after first write:", count_after_first)
print("Rows after retry:", count_after_retry)

In [0]:
assert count_after_retry == count_after_first

print("PASS: retry did not append the Aug 23 batch twice")

In [0]:
print("Input physical rows:", messages_count)
print("After first append:", count_after_first)
print("After retry:", count_after_retry)
print("Repeated message IDs:", duplicate_id_count)

assert messages_count == count_after_first
assert count_after_first == count_after_retry
assert duplicate_id_count > 0

print("\nPASS: Delta Bronze writer integration test")
print("PASS: schema aligned")
print("PASS: physical duplicates preserved")
print("PASS: retry idempotency works")
print("PASS: production Bronze untouched")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")
spark.sql(f"DROP SCHEMA IF EXISTS {TEST_SCHEMA}")

print("Temporary Bronze writer test objects removed")